<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/kikim6114/nlp2026/blob/main/06.Karpath Char LM-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

In [ ]:
# Colab이나 Kaggle을 사용하는 경우가 아니면, 이 cell을 모두 주석화하여 skip할 것.
import os

repository_name = 'nlp2026'
repository_url = f'https://github.com/kikim6114/{repository_name}.git'

# 항상 루트 경로(/content/)로 이동 후 확인
%cd /content/

if not os.path.exists(repository_name):
    !git clone {repository_url}
    print(f"{repository_name} 클론 완료")
else:
    print(f"{repository_name} 폴더가 이미 존재합니다. 클론을 건너뜁니다.")
%cd {repository_name}

<span style="font-size:3em; line-height:36px"><strong>RNNs and LSTMs for Character-Level Language Models
</strong></span>
- 이 자료는 Andrej Karpathy의 블로그 글 The Unreasonable Effectiveness of Recurrent eural Networks를 PyTorch로 구현한 것입니다.
- 원래 Karpathy는 Python과 Numpy만을 사용해서 모든 것을 scratch로 구현하였습니다.
- 여기서는 블로그에서 설명한 기본 모델만 실습합니다.

### Reference
- [The Unreasonable Effectiveness of Recurrent Neural Networks (Karpathy, 2015)](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)

In [16]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
%matplotlib inline 

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import urllib.request

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

### torch.Tensor 연습 

In [16]:
x = np.array(range(24), dtype=np.int32)
x.shape = (3, 2, 4)
xt = torch.Tensor(x)
xt.size()[-1]

4

In [2]:
def jj(x):
    return x, (2*x, 3*x)

In [7]:
y1, y2 = jj(1)
x2, x3 = y2
print(y1, x2, x3)

1 2 3


In [11]:
x = np.array(range(24), dtype=np.int32)
x.shape = (3, 2, 4)
x

array([[[ 0,  1,  2,  3],
        [ 4,  5,  6,  7]],

       [[ 8,  9, 10, 11],
        [12, 13, 14, 15]],

       [[16, 17, 18, 19],
        [20, 21, 22, 23]]])

In [9]:
x[0]

array([[0, 1, 2, 3],
       [4, 5, 6, 7]])

In [3]:
x[:, 0, :]

array([[ 0,  1,  2,  3],
       [ 8,  9, 10, 11],
       [16, 17, 18, 19]])

In [5]:
x[:, 1, :].shape

(3, 4)

In [7]:
x[:, -1, :]

array([[ 4,  5,  6,  7],
       [12, 13, 14, 15],
       [20, 21, 22, 23]])

In [8]:
x.ravel()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23])

In [15]:
y = x.reshape(1,1,-1)
y.squeeze().shape

(24,)

## Karpathy's character level language model
<img src="http://karpathy.github.io/assets/rnn/charseq.jpeg" width="400" height="400">

## Data Preprocessing

In [2]:
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
filename = 'shakespeare_data.txt'
urllib.request.urlretrieve(url, filename)

data_file = open(filename, 'r')
raw_data = data_file.read()
data_file.close()

print(raw_data[:200])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


- Vocabulary의 각 char를 index로 매핑
- Character를 one-hot encoding

In [3]:
data_length = len(raw_data)
vocab = sorted(list(set(raw_data)))
vocab_size = len(vocab)

char_to_index = { char:index for (index,char) in enumerate(vocab) }
index_to_char = { index:char for (index,char) in enumerate(vocab) }

print(f"데이터셋의 총 char 수 = {data_length}")
print()
print(f"Vocabulary = {vocab}")
print()
print(f"Vocabulary 크기 = {vocab_size}")
print()
print(f"char_to_index = {char_to_index}")
print()
print(f"index_to_char = {index_to_char}")

데이터셋의 총 char 수 = 1115394

Vocabulary = ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

Vocabulary 크기 = 65

char_to_index = {'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}

index_to_char = {0:

- 여기서는 데이터를 단순히 일정한 길이의 sub-sequence들로 잘라내어 사용하지만, 이 방법으로는 좋은 성능을 얻지 못할 수 있다. 

In [4]:
def create_one_hot(ind, length):
    """인덱스를 one-hot 벡터로 변환"""
    vec = np.zeros(length)
    vec[ind] = 1
    return vec

create_one_hot(2, 25)

array([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0.])

In [5]:
def chunk_data(raw_data, seq_len):
    """원시데이터를 같은 크기의 청크들로 분할"""
    chunks = []

    for i in range(len(raw_data) // seq_len):
        start = i * seq_len
        end = start + seq_len
        chunk = raw_data[start:end]
        # chunk += (10 - len(chunk))*'\n'
        chunks.append(chunk)
        
    return chunks

chunked_data = chunk_data(raw_data[:104], 25)
chunked_data

['First Citizen:\nBefore we ',
 'proceed any further, hear',
 ' me speak.\n\nAll:\nSpeak, s',
 'peak.\n\nFirst Citizen:\nYou']

In [6]:
def convert_dataset(dataset, char_to_index, vocab_size):
    """character sequence 데이터셋을 인덱스와 one-hot 데이터로 변환"""
    ind_dataset = np.zeros((len(dataset), len(dataset[0])), dtype=np.int32)
    one_hot_dataset = np.zeros((len(dataset), len(dataset[0]), vocab_size), dtype=np.float32)

    for i, seq in enumerate(dataset):
        ind_seq = [char_to_index[c] for c in seq]
        one_hot_seq = [create_one_hot(ind, vocab_size) for ind in ind_seq]
        
        ind_dataset[i, :] = np.array(ind_seq, dtype=np.float32)
        one_hot_dataset[i, :, :] = np.asarray(one_hot_seq, dtype=np.float32)

    return ind_dataset, one_hot_dataset
ind_dataset, one_hot_dataset = convert_dataset(chunked_data, char_to_index, vocab_size)
print(ind_dataset.shape)
print(one_hot_dataset.shape)

(4, 25)
(4, 25, 65)


one_hot_dataset = batch_size(4) x seq_length(25) x vector_size(65)

In [7]:
class ShakespeareDataset(Dataset):
    def __init__(self, inds, one_hot):
        self.inds = inds
        self.one_hot = one_hot

    def __len__(self):
        return self.one_hot.size(0)

    def __getitem__(self, idx):
        # 각 char의 target sequencer가 바로 다음 char의 seq가 되도록 offset시킴
        input_onehot = self.one_hot[idx, :-1, :]
        target_ind = self.inds[idx, 1:]

        return input_onehot, target_ind

In [8]:
CHUNK_LEN = 25

data_chunks = chunk_data(raw_data, CHUNK_LEN)
train_ind, train_oh = convert_dataset(data_chunks, char_to_index, vocab_size)
train_ind.shape, train_oh.shape

((44615, 25), (44615, 25, 65))

In [9]:
# Send data to GPU
train_ind_tt = torch.Tensor(train_ind).long().to(device)
train_oh_tt = torch.Tensor(train_oh).float().to(device)

train_set = ShakespeareDataset(train_ind_tt, train_oh_tt)

## Recurrent Neural Networks

$$ 
\begin{align} h_t &= W_{ih} x_t + W_{hh} h_{t-1} + b_{ih} + b_{hh}\\
 a_t &= \text{tanh}(h_t) \\
 o_t &= \text{softmax}(W_{ho} a_t + b_{ho}) 
 \end{align} 
$$
 
 
## Implementation

In [ ]:
class MyRNNCell(nn.Module):
    def __init__(self, obs_dim, hidden_size, output_dim):
        """RNN Cell 초기화"""
        super().__init__()
        self.hidden_size = hidden_size
        
        # 입력 x(t)와 은닉 h(t-1)를 merge하여 Linear층에서 처리.
        self.i2h = nn.Linear(obs_dim + hidden_size, hidden_size)  # x(t) & h(t-1) -> a(t)
        self.h2o = nn.Linear(hidden_size, output_dim)  # h(t) -> z(t)

        self.tanh = nn.Tanh()
        self.softmax = nn.LogSoftmax(dim=1)
    
    def forward(self, data, hidden):
        """RNN Cell의 forward pass 계산"""
        combined = torch.cat((data, hidden), 1)  # (64, 165)

        hidden = self.i2h(combined)
        hidden = self.tanh(hidden)

        output = self.h2o(hidden)
        output = self.softmax(output)

        return output, hidden
    

In [11]:
class MyRNN(nn.Module):
    def __init__(self, obs_dim, hidden_size, output_dim):
        """RNN 초기화"""
        super().__init__()
        self.hidden_size = hidden_size
        self.output_dim = output_dim

        self.rnn_cell = MyRNNCell(obs_dim, hidden_size, output_dim)

    def forward(self, x):
        """입력 sequence X에 대한 forward pass 수행.
        
        X의 shape = (B x L x D):
            B: batch size
            L: sequence length
            D: vocab_size
        """
        batch_size, seq_len, n_feat = x.size()  # (64, 24, 65)
        # [Q] seq_length가 처음 지정한 25에서 24로 왜 변했을까?
        # => DataLoader가 target seq 길이 24에 맞춰 자동 조정해줌
        
        # Stores outputs of RNN cell
        output_arr = torch.zeros((batch_size, seq_len, self.output_dim))    # (64, 24, 65)
        hidden_arr = torch.zeros((batch_size, seq_len, self.hidden_size))    # (64, 24, 100)
        
        # Send to GPU. This is a gotcha, make sure to send Tensors created
        # in a model to the same device as input Tensors.
        output_arr = output_arr.float().to(x.device)
        hidden_arr = hidden_arr.float().to(x.device)

        hidden = self.init_hidden(batch_size, x.device)  # (64, 100)

        for i in range(seq_len):
            # 각 순회마다 현재 위치 입력에 대해 RNN 계산
            # x[:, i, :]: (64, 24, 65) => (64, 65) :64개의 chunk들 각각의 i번째 char의 one-hot vector들
            output, hidden = self.rnn_cell(x[:, i, :], hidden) # input: ((64, 65), (64, 100))
            # [출력] output: (64, 65), hidden: (64, 100)
            
            output_arr[:, i, :] = output
            hidden_arr[:, i, :] = hidden

        return output_arr, hidden_arr

    def init_hidden(self, batch_size, device):
        """RNN hidden state 초기화"""
        return torch.zeros(batch_size, self.hidden_size, device=device)

## Training

In [12]:
def generate_seq(model, init_char_one_hot, length):
    """Sequence 생성
      1. RNN에서 다음 char의 분포를 구한다
      2. 이 분포를 사용하여 다음 char를 샘플링(예측)한다
      3. 샘플링된 char를 RNN에 입력으로 준다
      4. 위 절차를 반복한다
    """
    curr_char = init_char_one_hot  # (1, 1, 65)
    output = index_to_char[torch.argmax(curr_char.squeeze()).item()]  # (1,1,65) -> (65,) 한 다음 argmax 하여 얻은 정수값을 char로

    for i in range(length):
        out, _ = model(curr_char)  # curr_char: (1, 1, 65), out: (1, 1, 65)

        # 출력이 확률분포이므로, 이것으로부터 샘플을 생성할 수 있다
        p = np.exp(out[:, -1, :].cpu().detach().numpy())  # p: (1, 65)
        
        # 0 ~ (vacab_size-1) 까지의 정수 중에서 1 개 추출.
        # 확률분포 p는 ravel() 하여 (65,)인 벡터로 평활화한다
        # 파라미터 p를 지정하지 않으면 default 분포인 uniform distr으로 임의 표집
        out_ind = np.random.choice(range(vocab_size), p=p.ravel())
        
        out_char = index_to_char[out_ind]
        
        output += out_char
        
        # 예측된 char는 다음 시간스텝의 입력으로 사용된다.
        # float().to(device)를 빠뜨리지 않도록 주목할 것
        
        # curr_char의 one-hot vector 구하기
        curr_char = create_one_hot(out_ind, vocab_size)
        
        # one-hot vector 차원을 (1, 1, 65)로 변환
        curr_char = torch.Tensor(curr_char).float().to(device).view(1, 1, -1) 

    return output

In [ ]:
def train_loop(model, optimizer, train_loader, n_epochs, test_char=None):
    for epoch in range(n_epochs):
        avg_loss = []
        for input_seq, target_ind in train_loader:  # input_seq: (64, 24, 65), target_ind: (64, 24)
            optimizer.zero_grad()

            output, _ = model(input_seq)  # output: (64, 24, 65)
            loss = nn.NLLLoss()(output.transpose(1, 2), target_ind)  # 입력 ((64, 65, 24), (64, 24))

            loss.backward()
            
            # gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            
            optimizer.step()
            avg_loss.append(loss.item())

        print('\n*** Epoch {} : Avg Train Loss {}'.format(epoch, np.mean(avg_loss)))

        # 시퀀스 생성
        if test_char is not None:
            gen_seq = generate_seq(model, test_char, 100)
            print("Generated Sequence:\n {}".format(gen_seq))

In [14]:
HIDDEN_SIZE = 100
N_EPOCH = 10
LR = 0.01
BATCH_SIZE = 64
SAMP_CHAR = 'a'

# 앞의 train_set = ShakespeareDataset(train_ind_tt, train_oh_tt) 에서는 seq 길이가 25였지만
# target seq는 다음 글자로 offset되면서 길이가 24로 줄어든다.
# Dataloader에서 이것을 조정해서 training 입력 seq의 길이도 24로 줄여준다.
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)

model = MyRNN(vocab_size, HIDDEN_SIZE, vocab_size).to(device)
optim = torch.optim.Adam(model.parameters(), lr=LR)

test_char = create_one_hot(char_to_index[SAMP_CHAR], vocab_size)
test_char_tt = torch.Tensor(test_char).view(1, 1, -1).float().to(device)

train_loop(model, optim, train_loader, N_EPOCH, test_char_tt)

Epoch 0 : Avg Train Loss 2.1190645998733433
Generated Sequence:
 ake pl w thind, f tumongerund s?
Wed VI the Gre ino;
STO:
Chome s and ilil mak avinghay ath alid ajo 
Epoch 1 : Avg Train Loss 1.8447575464972794
Generated Sequence:
 anghes ysablord I:
APRises'ereeveronte thas; s, H
Hastom?
ANORIONEd overcha, my drrane yow wamistwe'Z
Epoch 2 : Avg Train Loss 1.7918278330718207
Generated Sequence:
 angomoume wa pinghandarmongimilerd, oorocaropo t bus woff med,
Go:

Thendg mave Pet s y t y asppo led
Epoch 3 : Avg Train Loss 1.7650042567348754
Generated Sequence:
 athin meeesellangeavee end greld Y h.
TO

GBRKETHorr.
ALol, misead f n onadath'thy.
LA wingan wid, d 
Epoch 4 : Avg Train Loss 1.7481096916349024
Generated Sequence:
 a yollindicoure S:
I:


Wadarpeprararoto t ifothoubus
G ithife m s I ara sse d y olelld,
Whu int t my
Epoch 5 : Avg Train Loss 1.7371152579613607
Generated Sequence:
 armma vinINCKENem ar. the d s d s marwhend
Y.
Fr thand,
Pser ind-de; s, t f gh tomby war'thad, n:
Wh

- 지금 모델로는 강의 슬라이드에서도 보았던 아래의 Karpathy의 원래 모델 출력은 기대할 수 없다.

>PANDARUS:
Alas, I think he shall be come approached and the day
When little srain would be attain'd into being never fed,
And who is but a chain and subjects of his death,
I should not sleep.

>Second Senator:
They are away this miseries, produced upon my soul,
Breaking and strongly should be buried, when I perish
The earth and thoughts of many states.

>DUKE VINCENTIO:
Well, your wit is in the care of side and that.

>Second Lord:
They would be ruled after this chamber, and
my fair nues begun out of the fact, to be conveyed,
Whose noble souls I'll have the heart of the wars.

>Clown:
Come, sir, I will make did behold your worship.

>VIOLA:
I'll drink it.

- LSTM이나 GRU 등을 이용하고 hidden layer를 여러 층 사용하는 방법을 사용하도록 구현해보자.

# 과제(Homework 2)
- 제출 마감: 2026.5.3 오후11:59:59
- 평가: 100점 기준에서 차감
### 과제 내용
- LSTM cell을 직접 작성한다. 즉, 앞서 실습한 **Implementation**처럼 PyTorch의 LSTM 함수를 이용하지 않고 scratch로 작성하여 수행한다.
- LSTM 층을 2개 사용한다.
- hidden_size = 100
- batch_size = 64
- Learning rate = 0.01
- N_EPOCH = 20
- sequence length = 30